In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import cv2
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_catm_b01.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4675 entries, 0 to 4674
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   participant_id        4675 non-null   object 
 1   brightness0           4675 non-null   float64
 2   brightness1           4675 non-null   float64
 3   brightness2           4675 non-null   float64
 4   brightness3           4675 non-null   float64
 5   local_symbol_cls0     4675 non-null   object 
 6   local_symbol_cls1     4675 non-null   object 
 7   local_symbol_cls2     4675 non-null   object 
 8   local_symbol_cls3     4675 non-null   object 
 9   correct_symbol_cls    4675 non-null   object 
 10  correct_local_cls     4675 non-null   int64  
 11  participant_linspace  4675 non-null   float64
 12  time_reaction_task    4675 non-null   int64  
 13  time_reaction_mean    4675 non-null   float64
 14  chosen_local_cls      4675 non-null   int64  
dtypes: float64(6), int64(

In [3]:
df

,participant_id,brightness0,brightness1,brightness2,brightness3,local_symbol_cls0,local_symbol_cls1,local_symbol_cls2,local_symbol_cls3,correct_symbol_cls,correct_local_cls,participant_linspace,time_reaction_task,time_reaction_mean,chosen_local_cls
0,Jupiter,0.181638,0.138126,0.138356,0.205347,y,c,f,m,c,1,0.000000,1029,980.0,2
1,Jupiter,0.220664,0.118452,0.176909,0.154541,s,f,m,j,f,1,0.000593,579,980.0,1
2,Jupiter,0.217307,0.125250,0.140104,0.199788,c,y,j,m,j,2,0.001186,688,980.0,3
3,Jupiter,0.132006,0.147323,0.113110,0.188276,f,s,m,c,c,3,0.001779,578,980.0,1
4,Jupiter,0.251793,0.169280,0.102143,0.135685,s,y,j,m,m,3,0.002372,848,980.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4670,Yunt,0.179335,0.168793,0.060612,0.254528,f,y,j,c,y,1,0.997557,1177,1424.0,1
4671,Yunt,0.152927,0.116505,0.119489,0.233521,m,c,y,s,m,0,0.998167,1132,1424.0,3
4672,Yunt,0.160991,0.234710,0.145027,0.171986,m,s,j,c,s,1,0.998778,1369,1424.0,2
4673,Yunt,0.214628,0.122164,0.117429,0.151460,m,y,j,c,y,1,0.999389,1623,1424.0,3


In [4]:
dataset_columns1 = ['participant_id', 'x1_linspace', 'x1_brightness',
                    'x1_symbol_m', 'x1_symbol_c', 'x1_symbol_s', 'x1_symbol_y', 'x1_symbol_f', 'x1_symbol_j',
                    'x1_location_0', 'x1_location_1', 'x1_location_2', 'x1_location_3',
                    'x2_linspace','x2_brightness',
                    'x2_symbol_m', 'x2_symbol_c', 'x2_symbol_s', 'x2_symbol_y', 'x2_symbol_f', 'x2_symbol_j',
                    'x2_location_0', 'x2_location_1', 'x2_location_2', 'x2_location_3',
]
df_dataset1 = pd.DataFrame(columns = dataset_columns1)
df_dataset1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   participant_id  0 non-null      object
 1   x1_linspace     0 non-null      object
 2   x1_brightness   0 non-null      object
 3   x1_symbol_m     0 non-null      object
 4   x1_symbol_c     0 non-null      object
 5   x1_symbol_s     0 non-null      object
 6   x1_symbol_y     0 non-null      object
 7   x1_symbol_f     0 non-null      object
 8   x1_symbol_j     0 non-null      object
 9   x1_location_0   0 non-null      object
 10  x1_location_1   0 non-null      object
 11  x1_location_2   0 non-null      object
 12  x1_location_3   0 non-null      object
 13  x2_linspace     0 non-null      object
 14  x2_brightness   0 non-null      object
 15  x2_symbol_m     0 non-null      object
 16  x2_symbol_c     0 non-null      object
 17  x2_symbol_s     0 non-null      object
 18  x2_symbol_y     0 non-

In [5]:
brightness_list = ['brightness0',	'brightness1',	'brightness2',	'brightness3']
local_smb_list = ['local_symbol_cls0',	'local_symbol_cls1',	'local_symbol_cls2',	'local_symbol_cls3']
x1_symbol_list = ['x1_symbol_m', 'x1_symbol_c', 'x1_symbol_s', 'x1_symbol_y', 'x1_symbol_f', 'x1_symbol_j']
x2_symbol_list = ['x2_symbol_m', 'x2_symbol_c', 'x2_symbol_s', 'x2_symbol_y', 'x2_symbol_f', 'x2_symbol_j']
x1_location_list = ['x1_location_0', 'x1_location_1', 'x1_location_2', 'x1_location_3' ]
x2_location_list = ['x2_location_0', 'x2_location_1', 'x2_location_2', 'x2_location_3' ]
symbols_list = ['m', 'c', 's', 'y', 'f', 'j']

In [6]:
idx=0
for index, row in df.iterrows():
  x1_location = row['chosen_local_cls']
  x1_linspace = row['participant_linspace']
  for b, l in zip(brightness_list , local_smb_list):
    if str(x1_location) in b:
      x1_brightness = row[b]
      x1_symbol = row[l]
  for _ in range(3):
    df_dataset1.loc[idx,'participant_id'] = row['participant_id']
    df_dataset1.loc[idx,'x1_linspace'] = x1_linspace
    df_dataset1.loc[idx,'x1_brightness'] = x1_brightness
    for sym_col in x1_symbol_list:
      if x1_symbol == sym_col[-1]:
        df_dataset1.loc[idx,sym_col] = 1.0
      else:
        df_dataset1.loc[idx,sym_col] = 0.0
    for loc_col in x1_location_list:
      if str(x1_location) == loc_col[-1]:
        df_dataset1.loc[idx,loc_col] = 1.0
      else:
        df_dataset1.loc[idx,loc_col] = 0.0
    idx += 1

df_dataset1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14025 entries, 0 to 14024
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   participant_id  14025 non-null  object
 1   x1_linspace     14025 non-null  object
 2   x1_brightness   14025 non-null  object
 3   x1_symbol_m     14025 non-null  object
 4   x1_symbol_c     14025 non-null  object
 5   x1_symbol_s     14025 non-null  object
 6   x1_symbol_y     14025 non-null  object
 7   x1_symbol_f     14025 non-null  object
 8   x1_symbol_j     14025 non-null  object
 9   x1_location_0   14025 non-null  object
 10  x1_location_1   14025 non-null  object
 11  x1_location_2   14025 non-null  object
 12  x1_location_3   14025 non-null  object
 13  x2_linspace     0 non-null      object
 14  x2_brightness   0 non-null      object
 15  x2_symbol_m     0 non-null      object
 16  x2_symbol_c     0 non-null      object
 17  x2_symbol_s     0 non-null      object
 18  x2_symbol_y

In [7]:
df_dataset1.head(9)

,participant_id,x1_linspace,x1_brightness,x1_symbol_m,x1_symbol_c,x1_symbol_s,x1_symbol_y,x1_symbol_f,x1_symbol_j,x1_location_0,...,x2_symbol_m,x2_symbol_c,x2_symbol_s,x2_symbol_y,x2_symbol_f,x2_symbol_j,x2_location_0,x2_location_1,x2_location_2,x2_location_3
0,Jupiter,0.0,0.138356,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jupiter,0.0,0.138356,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Jupiter,0.0,0.138356,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Jupiter,0.000593,0.118452,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Jupiter,0.000593,0.118452,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Jupiter,0.000593,0.118452,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Jupiter,0.001186,0.199788,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Jupiter,0.001186,0.199788,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Jupiter,0.001186,0.199788,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
idx=0
for index, row in df.iterrows():
  x2_linspace = row['participant_linspace']
  x1_location = row['chosen_local_cls']
  x2_brightness = []
  x2_symbol = []
  x2_location = []
  for b, l in zip(brightness_list , local_smb_list):
    if str(x1_location) not in b:
      x2_brightness.append(row[b])
      x2_symbol.append(row[l])
      x2_location.append(b[-1])
  for i in range(3):
    df_dataset1.loc[idx,'x2_linspace'] = x2_linspace
    df_dataset1.loc[idx,'x2_brightness'] = x2_brightness[i]
    #df_dataset1.loc[idx,'x2_symbol'] = x2_symbol[i]
    #df_dataset1.loc[idx,'x2_location'] = x2_location[i]
    for sym_col in x2_symbol_list:
      if x2_symbol[i] == sym_col[-1]:
        df_dataset1.loc[idx,sym_col] = 1.0
      else:
        df_dataset1.loc[idx,sym_col] = 0.0
    for loc_col in x2_location_list:
      if x2_location[i] == loc_col[-1]:
        df_dataset1.loc[idx,loc_col] = 1.0
      else:
        df_dataset1.loc[idx,loc_col] = 0.0
    idx += 1

df_dataset1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14025 entries, 0 to 14024
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   participant_id  14025 non-null  object
 1   x1_linspace     14025 non-null  object
 2   x1_brightness   14025 non-null  object
 3   x1_symbol_m     14025 non-null  object
 4   x1_symbol_c     14025 non-null  object
 5   x1_symbol_s     14025 non-null  object
 6   x1_symbol_y     14025 non-null  object
 7   x1_symbol_f     14025 non-null  object
 8   x1_symbol_j     14025 non-null  object
 9   x1_location_0   14025 non-null  object
 10  x1_location_1   14025 non-null  object
 11  x1_location_2   14025 non-null  object
 12  x1_location_3   14025 non-null  object
 13  x2_linspace     14025 non-null  object
 14  x2_brightness   14025 non-null  object
 15  x2_symbol_m     14025 non-null  object
 16  x2_symbol_c     14025 non-null  object
 17  x2_symbol_s     14025 non-null  object
 18  x2_symbol_y

In [9]:
df_dataset1.head(9)

,participant_id,x1_linspace,x1_brightness,x1_symbol_m,x1_symbol_c,x1_symbol_s,x1_symbol_y,x1_symbol_f,x1_symbol_j,x1_location_0,...,x2_symbol_m,x2_symbol_c,x2_symbol_s,x2_symbol_y,x2_symbol_f,x2_symbol_j,x2_location_0,x2_location_1,x2_location_2,x2_location_3
0,Jupiter,0.0,0.138356,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,Jupiter,0.0,0.138356,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,Jupiter,0.0,0.138356,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,Jupiter,0.000593,0.118452,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,Jupiter,0.000593,0.118452,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,Jupiter,0.000593,0.118452,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
6,Jupiter,0.001186,0.199788,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,Jupiter,0.001186,0.199788,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
8,Jupiter,0.001186,0.199788,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [10]:
df_dataset1.to_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref1_b01.csv', index=False)